In [16]:
import networkExpansionPy.folds as nf
import pandas as pd
from collections import Counter
from pathlib import PurePath, Path

# seed = sys.argv[1]
# random.seed(seed)
asset_path = nf.asset_path

# for vanilla
METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.01May2023.pkl") # path to metabolism object pickle
RN2RULES_PATH = PurePath(asset_path,"rn2fold","rn2rules.20230224.pkl") # path to rn2rules object pickle
SEED_CPDS_PATH = PurePath(asset_path, "compounds", "seeds.Goldford2022.csv") # path to seed compounds csv

# for FOLD-GATED
ALGORITHM = "no_look_ahead_rules"
WRITE = True # write result to disk
WRITE_TMP = False # write after each iteration
CUSTOM_WRITE_PATH = None # if writing result, custom path to write to
STR_TO_APPEND_TO_FNAME = None # if writing result, str to append to filename
ORDERED_OUTCOME = False # ignore random seed and always choose folds based on sort order
IGNORE_REACTION_VERSIONS = True # when maximizing for reactions, don't count versioned reactions

## Metabolism
metabolism = pd.read_pickle(METABOLISM_PATH)
# remove reactions that produce H2O2 before O2
H2O2_rns = ['R00017_v1', 'R03532_v1', 'R09507_v1', 'R09740_v1', 'R09741_v1', 'R11522', 'R12455', 'R12454']
condition = ((metabolism.network['rn'].isin(H2O2_rns)) & (metabolism.network['direction'] == 'reverse'))
metabolism.network = metabolism.network[~condition]
assert 'R00017_v1' not in list(metabolism.network['rn'])

## Ablation
metabolism.pruneReactionsFromMetabolite(['C00005'])
metabolism.pruneReactionsFromMetabolite(['Z00032'])
metabolism.pruneReactionsFromMetabolite(['C00006'])

## FoldRules
rn2rules = pd.read_pickle(RN2RULES_PATH)
foldrules = nf.FoldRules.from_rn2rules(rn2rules)

## Modify seeds with AA and GATP_rns
aa_cids = set(["C00037",
    "C00041",
    "C00065",
    "C00188",
    "C00183",
    "C00407",
    "C00123",
    "C00148",
    "C00049",
    "C00025"])

GATP_rns = {'R00200_gATP_v1',
    'R00200_gATP_v2',
    'R00430_gGTP_v1',
    'R00430_gGTP_v2',
    'R01523_gATP_v1',
    'R04144_gATP_v1',
    'R04208_gATP',
    'R04463_gATP',
    'R04591_gATP_v1',
    'R06836_gATP',
    'R06974_gATP',
    'R06975_gATP_v1'}


## Seed
seed = nf.Params(
    rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
    cpds = set((pd.read_csv(SEED_CPDS_PATH)["ID"])) | aa_cids,
    folds = set(['spontaneous'])
)

## Seed with pre-expansion
# preALL = pd.read_pickle('../data/pre-expansion_seed_cpds/pre-expansion_seed_cpds_ALL.pkl')
# seed = nf.Params(
#     rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#     cpds = set(preALL) | aa_cids,
#     folds = set(['spontaneous'])
# )

# ## Inititalize fold metabolism
fm = nf.FoldMetabolism(metabolism, foldrules, seed)

# ## run FOLD-GATED expansion
# result = fm.rule_order(algorithm=ALGORITHM, write=WRITE, write_tmp=WRITE_TMP, path=CUSTOM_WRITE_PATH, str_to_append_to_fname=STR_TO_APPEND_TO_FNAME, ordered_outcome=ORDERED_OUTCOME, ignore_reaction_versions=IGNORE_REACTION_VERSIONS)

calculating scope...


/Users/tseamuscorlett/Desktop/LongoLab/Fold_Gated/networkExpansion/networkExpansionPy/lib.py:535: SparseEfficiencyWarning: Comparing sparse matrices using == is inefficient, try using != instead.
  X,Y = netExp_trace(R,P,x0,b)


...done.


In [2]:
len(metabolism.network['rn'])

105933

# checking 'version # change' behavior

In [4]:
asset_path = nf.asset_path

METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.01May2023.pkl") # path to metabolism object pickle
df1 = pd.read_pickle(METABOLISM_PATH).network

METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.02Jun2024_peroxide_fix.pkl") # path to metabolism object pickle
df2 = pd.read_pickle(METABOLISM_PATH).network

In [51]:
df1[df1['rn'] == 'R00839_v2']

,rn,direction,cid,s,ub,lb,stoich_str,rn_old
4922,R00839_v2,forward,C04534,-1.0,0.1,1.000000e-07,NaN,R00839
4923,R00839_v2,forward,C00001,-1.0,0.1,1.000000e-07,NaN,R00839
4924,R00839_v2,forward,C00031,1.0,0.1,1.000000e-07,NaN,R00839
4925,R00839_v2,forward,C00092,1.0,0.1,1.000000e-07,NaN,R00839
0,R00839_v2,forward,Z00032,-1.0,0.1,1.000000e-07,NaN,R00839
1,R00839_v2,forward,Z00030,-1.0,0.1,1.000000e-07,NaN,R00839
0,R00839_v2,forward,Z00032,1.0,0.1,1.000000e-07,NaN,R00839
1,R00839_v2,forward,Z00030,1.0,0.1,1.000000e-07,NaN,R00839
4926,R00839_v2,reverse,C04534,1.0,0.1,1.000000e-07,NaN,R00839
4927,R00839_v2,reverse,C00001,1.0,0.1,1.000000e-07,NaN,R00839


In [52]:
df2[df2['rn'] == 'R00839_v2']

,rn,direction,cid,s,ub,lb,stoich_str,rn_old
4922,R00839_v2,forward,C04534,-1.0,0.1,1.000000e-07,NaN,R00839
4923,R00839_v2,forward,C00001,-1.0,0.1,1.000000e-07,NaN,R00839
4924,R00839_v2,forward,C00031,1.0,0.1,1.000000e-07,NaN,R00839
4925,R00839_v2,forward,C00092,1.0,0.1,1.000000e-07,NaN,R00839
0,R00839_v2,forward,Z00069,-1.0,0.1,1.000000e-07,NaN,R00839
0,R00839_v2,forward,Z00069,1.0,0.1,1.000000e-07,NaN,R00839
4926,R00839_v2,reverse,C04534,1.0,0.1,1.000000e-07,NaN,R00839
4927,R00839_v2,reverse,C00001,1.0,0.1,1.000000e-07,NaN,R00839
4928,R00839_v2,reverse,C00031,-1.0,0.1,1.000000e-07,NaN,R00839
4929,R00839_v2,reverse,C00092,-1.0,0.1,1.000000e-07,NaN,R00839


In [5]:
df2[df2['rn'] == 'R00839_v3']

,rn,direction,cid,s,ub,lb,stoich_str,rn_old
4922,R00839_v3,forward,C04534,-1.0,0.1,1.000000e-07,NaN,R00839
4923,R00839_v3,forward,C00001,-1.0,0.1,1.000000e-07,NaN,R00839
4924,R00839_v3,forward,C00031,1.0,0.1,1.000000e-07,NaN,R00839
4925,R00839_v3,forward,C00092,1.0,0.1,1.000000e-07,NaN,R00839
0,R00839_v3,forward,Z00032,-1.0,0.1,1.000000e-07,NaN,R00839
1,R00839_v3,forward,Z00030,-1.0,0.1,1.000000e-07,NaN,R00839
0,R00839_v3,forward,Z00032,1.0,0.1,1.000000e-07,NaN,R00839
1,R00839_v3,forward,Z00030,1.0,0.1,1.000000e-07,NaN,R00839
4926,R00839_v3,reverse,C04534,1.0,0.1,1.000000e-07,NaN,R00839
4927,R00839_v3,reverse,C00001,1.0,0.1,1.000000e-07,NaN,R00839


In [64]:
df1[df1['rn'] == 'R00946_v1']

,rn,direction,cid,s,ub,lb,stoich_str,rn_old
5656,R00946_v1,forward,C00440,-1.0,0.1,1.000000e-07,NaN,R00946
5657,R00946_v1,forward,C00155,-1.0,0.1,1.000000e-07,NaN,R00946
5658,R00946_v1,forward,C00101,1.0,0.1,1.000000e-07,NaN,R00946
5659,R00946_v1,forward,C00073,1.0,0.1,1.000000e-07,NaN,R00946
0,R00946_v1,forward,Z00018,-1.0,0.1,1.000000e-07,NaN,R00946
0,R00946_v1,forward,Z00018,1.0,0.1,1.000000e-07,NaN,R00946
5660,R00946_v1,reverse,C00440,1.0,0.1,1.000000e-07,NaN,R00946
5661,R00946_v1,reverse,C00155,1.0,0.1,1.000000e-07,NaN,R00946
5662,R00946_v1,reverse,C00101,-1.0,0.1,1.000000e-07,NaN,R00946
5663,R00946_v1,reverse,C00073,-1.0,0.1,1.000000e-07,NaN,R00946


In [65]:
df2[df2['rn'] == 'R00946_v1']

,rn,direction,cid,s,ub,lb,stoich_str,rn_old
5656,R00946_v1,forward,C00440,-1.0,0.1,1.000000e-07,NaN,R00946
5657,R00946_v1,forward,C00155,-1.0,0.1,1.000000e-07,NaN,R00946
5658,R00946_v1,forward,C00101,1.0,0.1,1.000000e-07,NaN,R00946
5659,R00946_v1,forward,C00073,1.0,0.1,1.000000e-07,NaN,R00946
0,R00946_v1,forward,Z00018,-1.0,0.1,1.000000e-07,NaN,R00946
0,R00946_v1,forward,Z00018,1.0,0.1,1.000000e-07,NaN,R00946
5660,R00946_v1,reverse,C00440,1.0,0.1,1.000000e-07,NaN,R00946
5661,R00946_v1,reverse,C00155,1.0,0.1,1.000000e-07,NaN,R00946
5662,R00946_v1,reverse,C00101,-1.0,0.1,1.000000e-07,NaN,R00946
5663,R00946_v1,reverse,C00073,-1.0,0.1,1.000000e-07,NaN,R00946


# making 'rns_unreached_wo_cofactor' pkl files

In [3]:
import pickle
full_rns_scope = fm.scope.rns

In [17]:
print(len(full_rns_scope), len(fm.scope.rns))

my_set = full_rns_scope - fm.scope.rns
print(len(my_set))

7678 3419
4259


In [18]:
with open('../rns_unreached_wo_NADP_NADPH_zNADP.pkl', 'wb') as file:
    pickle.dump(my_set, file)

# what are the 3 compounds that fall off?

In [ ]:
len(fm.scope.cpds), len(set((pd.read_csv(SEED_CPDS_PATH)["ID"]))), len(seed.cpds)

In [ ]:
run_seeds = []
for c, i in fm.scope.cpd_iteration_dict.items():
    if i == 0:
        run_seeds.append(c)
len(run_seeds)

In [ ]:
for c in ['C00034', 'Z00020', 'C00050']:
    print(c, cpd2name[c], ':', c in seed.cpds, c in fm.scope.cpds)

# analysis

In [ ]:
import re
import csv
import ast
import matplotlib.pyplot as plt
import pandas as pd
import random
from bokeh.plotting import figure, output_file, show
from bokeh.models import HoverTool

def todata(dict1, dict2, val_type = 'MEAN'):
    valid_keys = list(dict1.keys() & dict2.keys())
    data1 = [dict1[x] for x in valid_keys]
    data2 = [dict2[x] for x in valid_keys]
    
    if type(data1[0]) == dict:
        data1 = [x[val_type] for x in data1]
        
    if type(data2[0]) == dict:
        data2 = [x[val_type] for x in data2]
    
    return valid_keys, data1, data2

def dict2csv(my_dict, csv_file_path):
    with open(csv_file_path, 'w', newline='') as csv_file:
        csv_writer = csv.writer(csv_file)
        for key, value in my_dict.items():
            csv_writer.writerow([key] + [value])

def csv2dict(csv_file_path):
    result_dict = {}
    with open(csv_file_path, 'r') as csv_file:
        csv_reader = csv.reader(csv_file)
        
        for row in csv_reader:
            try:
                result_dict[row[0]] = ast.literal_eval(row[1])  # value is list
            except:
                result_dict[row[0]] = row[1]
                
    return result_dict

def scatter(dict1, dict2, x_axis = 'x-axis', y_axis = 'y-axis'):
    valid_keys, data1, data2 = todata(dict1, dict2)
    plt.scatter(data1, data2, marker='o', color='b', alpha = 0.1, label='Data Points', zorder=2)
    plt.xlabel(x_axis)
    plt.ylabel(y_axis)
    # plt.savefig('scatter.png')
    plt.show()

In [ ]:
module2name = csv2dict('../data/assets/module2name.csv')
module2rns = csv2dict('../data/assets/module2rns.csv')

rn2modules = csv2dict('../data/assets/rn2modules.csv')
rn2def = csv2dict('../data/assets/rn2def_versions.csv')
rn2rules = pd.read_pickle('../data/assets/rn2rules.20230224.pkl')
rn2eqn_SI = csv2dict('../data/assets/rn2eqn_SI.csv')
cpd2name = csv2dict('../data/assets/cpd2nameShort.csv')

# vanilla

In [ ]:
vanilla_iter2cpds = {}
for c, i in fm.scope.cpd_iteration_dict.items():
    if i not in vanilla_iter2cpds.keys():
        vanilla_iter2cpds[i] = [c]
    else:
        vanilla_iter2cpds[i].append(c)
        
vanilla_iter2cpdsNum = {}
for i in vanilla_iter2cpds.keys():
    vanilla_iter2cpdsNum[i] = len(vanilla_iter2cpds[i])
    
plt.figure(figsize=(30, 5))
plt.plot(vanilla_iter2cpdsNum.values(), color='k')
plt.xlim([0, 85])
plt.ylim([0, 260])
plt.xticks(fontsize='18')
plt.yticks(fontsize='18')
# plt.savefig('vanilla_iter_vs_cpds_10AA.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# annotate cpds

cpd2iter = {}
for cpd in ['C0007', 'C00018', 'C00302', 'C00334', 'C11440', 'C00002', 'C00003', 'C00010', 'C00024', 'C00129', 'C00068', 'C00019', 'C00101', 'C00016', 'C00061', 'C00363', 'C00194', 'C00062', 'C00007', 'C00448', 'C11901', 'C00341', 'C00353', 'C00923', 'C00463', 'C00229', 'C05819', 'C03313', 'C00027', 'C00032', 'C00059', 'C00094', 'C05773', 'C00251', 'C00082', 'C00079', 'C00078', 'C02059', 'C00828', 'C01063', 'C00120']:
    for i, cs in vanilla_iter2cpds.items():
        for c in cs:
            if c == cpd:
                cpd2iter[c] = i

In [ ]:
plt.figure(figsize=(30, 5))
plt.plot(vanilla_iter2cpdsNum.values(), color='k')

# annotate cpds

for c in cpd2iter:
    y = random.randint(0, 250)
    plt.text(cpd2iter[c], y, cpd2name[c], ha='center', va='bottom', color='red')
    plt.scatter(cpd2iter[c], y-5, color='red')

# plt.savefig('3B_cpds_vanilla.svg', dpi=300, bbox_inches='tight')
plt.show()

# fold-gated

In [ ]:
result = pd.read_pickle('../runs/2024-04-25_11-00-45_no_look_ahead_rules_ignore_versions_3141_cumiter_fix.pkl.gz')

In [ ]:
folditer2cpd = {}
for c, order in result.cpds_folditer.items():
    if order not in folditer2cpd.keys():
        folditer2cpd[order] = [c]
    else:
        folditer2cpd[order].append(c)
print(len(folditer2cpd))

folditer2cpdNum = {}
for fold in folditer2cpd.keys():
    folditer2cpdNum[fold] = len(folditer2cpd[fold])

folditer2cpdNumFull = {}
for i in range(0, max(folditer2cpd.keys())):
    if i not in folditer2cpdNum.keys():
        folditer2cpdNumFull[i] = 0
    else:
        folditer2cpdNumFull[i] = folditer2cpdNum[i]
        
plt.figure(figsize=(30, 5))
plt.plot(folditer2cpdNumFull.values(), color='k')

plt.xlim([0, 360])
# plt.savefig('3B_cpds.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# from pre-expansion
cpds = ['Z00035', 'C00002', 'C00004', 'C00019', 'C00010', 'C00016', 'Z00047', 'Z00009']
# from vanilla (+ FMN, AcetylCoA, isopentenyl diphosphate, oxygen)
cpds = ['C00007','C00024','C00061','C00129','Z00014', 'Z00035', 'C00002', 'C00004', 'C00019', 'C00010', 'C00016', 'Z00047', 'Z00009']

dict1 = result.cpds_folditer
dict2 = fm.scope.cpd_iteration_dict

# if you don't care about zorder of red vs. blue
# valid_keys, data1, data2 = todata(dict1, dict2)
# colors = ['red' if c in cpds else 'blue' for c in valid_keys]
# plt.scatter(data1, data2, marker='o', color=colors, alpha = 0.5, label='Data Points', zorder=2)

# if you red in front of blue
booleans = [c in cpds for c in valid_keys]
data1_bool = [value for value, flag in zip(data1, booleans) if flag]
data2_bool = [value for value, flag in zip(data2, booleans) if flag]
plt.scatter(data1, data2, marker='o', color='blue', alpha = 0.1, label='Data Points', zorder=2)
plt.scatter(data1_bool, data2_bool, marker='o', color='red', alpha = 0.5, label='Data Points', zorder=3)

# annotate cpds
for c in cpds:
    plt.text(dict1[c], dict2[c], cpd2name[c], ha='center', va='bottom', color='red')

plt.ylabel('vanilla')
plt.xlabel('fold_gated')
# plt.savefig('scatter_vanilla_vs_fold_gated_key_cpds.svg')
plt.show()

In [ ]:
# bokeh

output_file("vanilla_vs_fold_gated_order.html")
p = figure(plot_width=800, plot_height=800, title="vanilla_vs_fold_gated_order")

# Add annotations
# source = {'x': list(group12iter.values()), 'y': y_values, 'label': ['X' + str(key) for key in group12iter.keys()]}
dict1 = {key:value for key, value in result.cpds_folditer.items() if key not in cpds}
dict2 = {key:value for key, value in fm.scope.cpd_iteration_dict.items() if key not in cpds}
valid_keys, data1, data2 = todata(dict1, dict2)

dict1_red = {key:value for key, value in result.cpds_folditer.items() if key in cpds}
dict2_red = {key:value for key, value in fm.scope.cpd_iteration_dict.items() if key in cpds}
valid_keys_red, data1_red, data2_red = todata(dict1_red, dict2_red)

source = {'x': data1, 'y': data2, 'label': [cpd2name.get(key, 'no name') for key in valid_keys]}
p.scatter('x', 'y', source=source, size=10, color='blue', alpha=0.3)

source = {'x': data1_red, 'y': data2_red, 'label': [cpd2name.get(key, 'no name') for key in valid_keys_red]}
p.scatter('x', 'y', source=source, size=10, color='red', alpha=0.3)

# Add hover tool
hover = HoverTool()
hover.tooltips = [("folditer", "@x"), ("cpd", "@label")]
p.add_tools(hover)

# Customize plot
p.xaxis.axis_label = 'fold_gated'
p.yaxis.axis_label = 'vanilla'
p.xaxis.ticker = [50, 100, 150, 200, 250, 300, 350]  # Convert range to list
p.xgrid.grid_line_color = None

# Show the plot
show(p)

# carbon fixation pathways: validating result from vanilla

In [ ]:
cm = ['M00165', 'M00173', 'M00374', 'M00375', 'M00376', 'M00377']

In [ ]:
cm2rn = {}

for m in cm:
    print(m, module2name[m])
    print(module2rn[m])
    cm2rn[m] = set(module2rn[m])
    
    # check if any module is satisfied in vanilla
    print(all(item in [item[:6] for item in fm.scope.rns] for item in module2rn[m]))
    
    print('')

In [ ]:
iter2rn = {}

for rn, i in fm.scope.rn_iteration_dict.items():
    if i not in iter2rn.keys():
        iter2rn[i] = [rn]
    else:
        iter2rn[i].append(rn)

In [ ]:
cm = ['M00165', 'M00173', 'M00374', 'M00375', 'M00376', 'M00377']
rn_list = []

for i, rns in iter2rn.items():
    # keep updating rn_list
    rn_list.extend(rns)
    
    # for each cm module, check if all rns in module are in rn_list
    for m in cm:
        if all(item in [item[:6] for item in rn_list] for item in module2rn[m]):
            print(f'{m}: {module2name[m]} is feasible at {i}')
            cm.remove(m)

## with fold-gating

In [ ]:
withP = pd.read_pickle('../2024-04-19_19-53-04_with_protein.pkl.gz')

In [ ]:
withP.iteration_cum

In [ ]:
iter2rn = {}

for rn, i in withP.rns_cumiter.items():
    if i not in iter2rn.keys():
        iter2rn[i] = [rn]
    else:
        iter2rn[i].append(rn)

In [ ]:
cm = ['M00165', 'M00173', 'M00374', 'M00375', 'M00376', 'M00377']
rn_list = []

for i, rns in iter2rn.items():
    # keep updating rn_list
    rn_list.extend(rns)
    
    # for each cm module, check if all rns in module are in rn_list
    for m in cm:
        if all(item in [item[:6] for item in rn_list] for item in module2rn[m]):
            print(f'{m}: {module2name[m]} is feasible at {i}')
            cm.remove(m)

# checking cofactor dependence: two directions

In [ ]:
import csv
import ast

def csv2dict(csv_file_path):
    result_dict = {}
    with open(csv_file_path, 'r') as csv_file:
        csv_reader = csv.reader(csv_file)
        
        for row in csv_reader:
            try:
                result_dict[row[0]] = ast.literal_eval(row[1])  # value is list
            except:
                result_dict[row[0]] = row[1]
                
    return result_dict

rn2cpds_SI = csv2dict('../data/assets/rn2cpds_SI.csv')
rn2def_SI = csv2dict('../data/assets/rn2def_complete.csv')
cpd2name = csv2dict('../data/assets/cpd2nameShort.csv')

In [ ]:
cpd2name['C00003'], cpd2name['C01352']

In [ ]:
# FAD+/FADH2

for rn, cpds in rn2cpds_SI.items():
    if 'C00016' in cpds:
        if 'C01352' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))
print('-----')
for rn, cpds in rn2cpds_SI.items():
    if 'C01352' in cpds:
        if 'C00016' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))

In [ ]:
# NAD+/NADH

for rn, cpds in rn2cpds_SI.items():
    if 'C00003' in cpds:
        if 'C00004' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))
print('-----')
for rn, cpds in rn2cpds_SI.items():
    if 'C00004' in cpds:
        if 'C00003' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))

In [ ]:
# NADP+/NADPH

for rn, cpds in rn2cpds_SI.items():
    if 'C00005' in cpds:
        if 'C00006' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))
print('-----')
for rn, cpds in rn2cpds_SI.items():
    if 'C00006' in cpds:
        if 'C00005' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))

In [ ]:
# ATP/ADP

for rn, cpds in rn2cpds_SI.items():
    if 'C00002' in cpds:
        if 'C00008' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))
print('-----')
for rn, cpds in rn2cpds_SI.items():
    if 'C00008' in cpds:
        if 'C00002' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))

In [ ]:
# SAM/SAH

for rn, cpds in rn2cpds_SI.items():
    if 'C00019' in cpds:
        if 'C00021' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))
print('-----')
for rn, cpds in rn2cpds_SI.items():
    if 'C00021' in cpds:
        if 'C00019' not in cpds:
            print(rn, cpds, rn2def_SI.get(rn[:6], 0))

# compare before 'protein' fix vs. after fix

In [ ]:
vanilla_70seed = csv2dict('../data/assets/vanilla_cpd2iter_70seed.csv')
vanilla_before = csv2dict('../data/assets/vanilla_cpd2iter_before.csv')
vanilla_after = csv2dict('../data/assets/vanilla_cpd2iter_after.csv')

In [ ]:
vanilla_iter2cpds_70 = {}
for c, i in vanilla_70seed.items():
    if i not in vanilla_iter2cpds_70.keys():
        vanilla_iter2cpds_70[i] = [c]
    else:
        vanilla_iter2cpds_70[i].append(c)
        
vanilla_iter2cpds_70Num = {}
for i in vanilla_iter2cpds_70.keys():
    vanilla_iter2cpds_70Num[i] = len(vanilla_iter2cpds_70[i])
    
plt.figure(figsize=(10, 5))
plt.plot(vanilla_iter2cpds_70Num.values(), color='green')
# plt.savefig('3B_cpds_vanilla.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# from pre-expansion
cpds = ['Z00035', 'C00002', 'C00004', 'C00019', 'C00010', 'C00016', 'Z00047', 'Z00009']

# from vanilla (+ FMN, AcetylCoA, isopentenyl diphosphate, oxygen)
cpds = ['C00007','C00024','C00061','C00129','Z00014', 'Z00035', 'C00002', 'C00004', 'C00019', 'C00010', 'C00016', 'Z00047', 'Z00009']

In [ ]:
# cummulative growth 70 seed

cpds = ['C00007','C00024','C00061','C00129','Z00014', 'Z00035', 'C00002', 'C00004', 'C00019', 'C00010', 'C00016', 'Z00047']
vanilla_iter2cpds_70Num_cumm = {}
cumm = 0
for i, num in vanilla_iter2cpds_70Num.items():
    vanilla_iter2cpds_70Num_cumm[i] = num + cumm
    cumm += num
vanilla_iter2cpds_70Num_cumm

plt.figure(figsize=(6, 5))
plt.plot(vanilla_iter2cpds_70Num_cumm.values(), color='k')

# annotate cpds
for c in cpds:
    x = vanilla_70seed[c]
    y = vanilla_iter2cpds_70Num_cumm[x]
    plt.text(x+7, y-100, cpd2name[c], ha='center', va='bottom', color='red')
    plt.scatter(x, y, marker='o', color='red', alpha = 0.5, label='Data Points', zorder=2)
    
plt.xticks(fontsize='14')
plt.yticks(fontsize='14')
plt.xlabel('Expansion iteration', fontsize='14')
plt.ylabel('Network Size', fontsize='14')

# plt.savefig('vanilla_iter_vs_cumm_cpds_annotated.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# cummulative growth 80 seed

cpds = ['C00007','C00024','C00061','C00129','Z00014', 'Z00035', 'C00002', 'C00004', 'C00019', 'C00010', 'C00016', 'Z00047']
vanilla_iter2cpds_before_cumm = {}
cumm = 0
for i, num in vanilla_iter2cpds_beforeNum.items():
    vanilla_iter2cpds_before_cumm[i] = num + cumm
    cumm += num
vanilla_iter2cpds_before_cumm

plt.figure(figsize=(6, 5))
plt.plot(vanilla_iter2cpds_before_cumm.values(), color='k')

# annotate cpds
for c in cpds:
    x = vanilla_before[c]
    y = vanilla_iter2cpds_before_cumm[x]
    plt.text(x+7, y-100, cpd2name[c], ha='center', va='bottom', color='red')
    plt.scatter(x, y, marker='o', color='red', alpha = 0.5, label='Data Points', zorder=2)
    
plt.xticks(fontsize='14')
plt.yticks(fontsize='14')
plt.xlabel('Expansion iteration', fontsize='14')
plt.ylabel('Network Size', fontsize='14')

# plt.savefig('vanilla_iter_vs_cumm_cpds_80seed_annotated.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
vanilla_iter2cpds_after = {}
for c, i in vanilla_after.items():
    if i not in vanilla_iter2cpds_after.keys():
        vanilla_iter2cpds_after[i] = [c]
    else:
        vanilla_iter2cpds_after[i].append(c)
        
vanilla_iter2cpds_afterNum = {}
for i in vanilla_iter2cpds_after.keys():
    vanilla_iter2cpds_afterNum[i] = len(vanilla_iter2cpds_after[i])
    
plt.figure(figsize=(30, 5))
plt.plot(vanilla_iter2cpds_afterNum.values(), color='k')
# plt.savefig('3B_cpds_vanilla.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
vanilla_iter2cpds_before = {}
for c, i in vanilla_before.items():
    if i not in vanilla_iter2cpds_before.keys():
        vanilla_iter2cpds_before[i] = [c]
    else:
        vanilla_iter2cpds_before[i].append(c)
        
vanilla_iter2cpds_beforeNum = {}
for i in vanilla_iter2cpds_before.keys():
    vanilla_iter2cpds_beforeNum[i] = len(vanilla_iter2cpds_before[i])

plt.figure(figsize=(30, 5))
plt.plot(vanilla_iter2cpds_beforeNum.values(), color='k')
# plt.savefig('3B_cpds_vanilla.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
scatter(vanilla_after, vanilla_before, 'after', 'before')

In [ ]:
dict1 = vanilla_after
dict2 = vanilla_before

valid_keys, data1, data2 = todata(dict1, dict2)

# if you don't care about zorder of blue vs. red
# colors = ['red' if c in cpds else 'blue' for c in valid_keys]
# plt.scatter(data1, data2, marker='o', color=colors, alpha = 0.1, label='Data Points', zorder=2)

# if you want red to be in front
plt.scatter(data1, data2, marker='o', color='blue', alpha = 0.1, label='Data Points', zorder=2)
booleans = [c in cpds for c in valid_keys]
data1_bool = [value for value, flag in zip(data1, booleans) if flag]
data2_bool = [value for value, flag in zip(data2, booleans) if flag]
plt.scatter(data1_bool, data2_bool, marker='o', color='red', alpha = 0.5, label='Data Points', zorder=2)

# annotate cpds
for c in cpds:
    plt.text(vanilla_after[c], vanilla_before[c], cpd2name[c], ha='center', va='bottom', color='red')

plt.xlabel('after fix')
plt.ylabel('before fix')
# plt.savefig('scatter_vanilla_before_vs_after.svg')
plt.show()

In [ ]:
aa = ['C00017', 'C00037', 'C00077', 'C00041', 'C00097', 'C00049', 'C00025', 'C00079', 'C00135', 'C00407', 'C00047', 'C00123', 'C00073', 'C00152', 'C00148', 'C00064', 'C00062', 'C00065', 'C00188', 'C00183', 'C00078', 'C00082']

dict1 = vanilla_after
dict2 = vanilla_before

valid_keys, data1, data2 = todata(dict1, dict2)
# colors = ['red' if c in cpds else 'blue' for c in valid_keys]
# plt.scatter(data1, data2, marker='o', color=colors, alpha = 0.1, label='Data Points', zorder=2)

plt.scatter(data1, data2, marker='o', color='blue', alpha = 0.1, label='Data Points', zorder=2)
booleans = [c in aa for c in valid_keys]
data1_bool = [value for value, flag in zip(data1, booleans) if flag]
data2_bool = [value for value, flag in zip(data2, booleans) if flag]
plt.scatter(data1_bool, data2_bool, marker='o', color='red', alpha = 0.5, label='Data Points', zorder=2)

# annotate cpds
for c in aa:
    plt.text(vanilla_after[c], vanilla_before[c], cpd2name[c], ha='center', va='bottom', color='red')

plt.xlabel('after fix')
plt.ylabel('before fix')
# plt.savefig('scatter_vanilla_before_vs_after_AA.svg')
plt.show()

In [ ]:
# bokeh

output_file("annotated_plot.html")
p = figure(plot_width=800, plot_height=800, title="cpd order (after) vs. cpd order (before)")

# Add annotations
# source = {'x': list(group12iter.values()), 'y': y_values, 'label': ['X' + str(key) for key in group12iter.keys()]}
key, data1, data2 = todata(vanilla_after, vanilla_before)
source = {'x': data1, 'y': data2, 'label': [cpd2name.get(key, 'no name') for key in key]}
p.scatter('x', 'y', source=source, size=10, color='blue')

# Add hover tool
hover = HoverTool()
hover.tooltips = [("folditer", "@x"), ("cpd", "@label")]
p.add_tools(hover)

# Customize plot
p.xaxis.axis_label = 'cpd order (after)'
p.yaxis.axis_label = 'cpd order (before)'
p.xaxis.ticker = [50, 100, 150, 200, 250, 300, 350]  # Convert range to list
p.xgrid.grid_line_color = None

# Show the plot
show(p)

# ~50% of seed reactions are 'lost'!

In [ ]:
len(fm.seed.cpds)

In [ ]:
len(fm.seed.rns - fm.scope.rns)  # number of rns in seed that the algo never reaches

In [ ]:
len(fm.seed.rns), len(fm.scope.rns), len(fm.seed.rns & fm.scope.rns)

# get seed sets for pre-expansion

In [ ]:
preexp_cpds = ['Z00035', 'C00002', 'C00004', 'C00019', 'C00010', 'C00016', 'Z00047', 'Z00009']

cpd2seeds = {}
cpd2seeds['NONE'] = fm.seed.cpds

for cpd in preexp_cpds:
    seeds = []
    cpditer = fm.scope.cpd_iteration_dict[cpd]
    for c, i in fm.scope.cpd_iteration_dict.items():
        if i <= cpditer:
            seeds.append(c)
    cpd2seeds[cpd] = seeds

cpd2seeds['ALL'] = fm.scope.cpds

In [ ]:
for c, seeds in cpd2seeds.items():
    print(c, len(seeds))

In [ ]:
# for cpd, seeds in cpd2seeds.items():
#     with open(f'pre-expansion_seed_cpds/pre-expansion_seed_cpds_{cpd}.pkl', 'wb') as file:
#         pickle.dump(seeds, file)

In [ ]:
a = pd.read_pickle('pre-expansion_seed_cpds/pre-expansion_seed_cpds_NONE.pkl')
print(a)

# investigating photosyn_R09503_vX

In [ ]:
rn2eqn_SI['R09503_vX']

In [ ]:
rn2eqn_SI['R09503_v1']

In [ ]:
# according to Liam: new reactions were added to account for the possibility that ancient reactions 
# could be catalyzed with prebiotic, geochemical context; so these reactions are NOT meant to be the same; 
# it's ok that some of them are not fold-gated

# script for shuffle1

In [ ]:
# shuffle2 was done with multiprocessing from PyCharm

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import networkExpansionPy.folds as nf
import pandas as pd
from pathlib import PurePath
import random

def shuffle_dict_pairs(d):
    # Extract keys and values
    keys = list(d.keys())
    values = list(d.values())

    # Shuffle the keys
    random.shuffle(keys)

    # Create a new dictionary with shuffled key-value pairs
    shuffled_dict = dict(zip(keys, values))

    return shuffled_dict


if __name__ == '__main__':
    preNONE = pd.read_pickle(
        '../fold_pre-expansion-results.26Sep2023/2023-09-26_19-25-59_rules_to_rn_randomSeed_42_preExpansion_NONE.pkl.gz')
    preATP = pd.read_pickle(
        '../fold_pre-expansion-results.26Sep2023/2023-09-26_20-37-47_rules_to_rn_randomSeed_42_preExpansion_C00002_iter20.pkl.gz')
    print(len(set(preATP.cpds_subiter[0])))

    asset_path = nf.asset_path

    # for vanilla
    METABOLISM_PATH = PurePath(asset_path, "metabolic_networks",
                               "metabolism.v8.02Jun2024_peroxide_fix.pkl")  # path to metabolism object pickle
    RN2RULES_PATH = PurePath(asset_path, "rn2fold", "rn2rules.20230224.pkl")  # path to rn2rules object pickle
    SEED_CPDS_PATH = PurePath(asset_path, "compounds", "seeds.Goldford2022.csv")  # path to seed compounds csv

    # for FOLD-GATED
    ALGORITHM = "no_look_ahead_rules"
    WRITE = True  # write result to disk
    WRITE_TMP = False  # write after each iteration
    CUSTOM_WRITE_PATH = None  # if writing result, custom path to write to
    ORDERED_OUTCOME = False  # ignore random seed and always choose folds based on sort order
    IGNORE_REACTION_VERSIONS = True  # when maximizing for reactions, don't count versioned reactions

    ## Metabolism
    metabolism = pd.read_pickle(METABOLISM_PATH)

    ## FoldRules
    rn2rules = pd.read_pickle(RN2RULES_PATH)
    RANDOMSEED = 33671
    random.seed(RANDOMSEED)
    # set file name
    STR_TO_APPEND_TO_FNAME = "shuffle1_%s" % RANDOMSEED

    # Inititalize fold metabolism with shuffling
    rn2rules_shuffle = shuffle_dict_pairs(rn2rules)  # Shuffle the dictionary pairs
    foldrules = nf.FoldRules.from_rn2rules(rn2rules_shuffle)
    fm = nf.FoldMetabolism(metabolism, foldrules, seed)

    ## Modify seeds with AA and GATP_rns
    aa_cids = set(["C00037",
                   "C00041",
                   "C00065",
                   "C00188",
                   "C00183",
                   "C00407",
                   "C00123",
                   "C00148",
                   "C00049",
                   "C00025"])

    GATP_rns = {'R00200_gATP_v1',
                'R00200_gATP_v2',
                'R00430_gGTP_v1',
                'R00430_gGTP_v2',
                'R01523_gATP_v1',
                'R04144_gATP_v1',
                'R04208_gATP',
                'R04463_gATP',
                'R04591_gATP_v1',
                'R06836_gATP',
                'R06974_gATP',
                'R06975_gATP_v1'}

    # Seed (modify with pre_expansion seed)
    seed = nf.Params(
        rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,
        cpds = set(preATP.cpds_subiter[0]) | aa_cids,
        folds = set(['spontaneous'])
    )

    ## Seed (no pre-expansion)
    # seed = nf.Params(
    #     rns=set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with non-fold-gated reactions
    #     cpds=set((pd.read_csv(SEED_CPDS_PATH)["ID"])) | aa_cids,
    #     folds=set(['spontaneous'])
    # )

## Inititalize fold metabolism
fm = nf.FoldMetabolism(metabolism, foldrules, seed)

# run FOLD-GATED expansion
result = fm.rule_order(algorithm=ALGORITHM, write=WRITE, write_tmp=WRITE_TMP, path=CUSTOM_WRITE_PATH, str_to_append_to_fname=STR_TO_APPEND_TO_FNAME, ordered_outcome=ORDERED_OUTCOME, ignore_reaction_versions=IGNORE_REACTION_VERSIONS)
